# Resume Ranker — Qwen2.5-3B QLoRA Fine-tune (Lightning.ai)
**Model**: `unsloth/Qwen2.5-3B-Instruct`  
**Task**: (JD + resume_text) → candidate profile + score_breakdown + relevance_score + rationale  
**Hardware**: Lightning.ai T4/A10G  
**Expected training time**: ~50–70 min on T4, ~25–35 min on A10G

### Before running:
1. Upload `sft_pointwise.jsonl` via the Lightning.ai file browser (left sidebar → Files)
2. Place it at `/teamspace/studios/this_studio/sft_pointwise.jsonl`
3. Set runtime to GPU (T4 or A10G)

## 1. Install dependencies

In [ ]:
%%capture
!pip install unsloth
!pip install --upgrade transformers datasets accelerate peft trl bitsandbytes

## 2. Set data path
File should already be uploaded via Lightning file browser.

In [ ]:
import os

DATA_PATH   = "/teamspace/studios/this_studio/sft_pointwise.jsonl"
OUTPUT_DIR  = "/teamspace/studios/this_studio/qwen_resume_ranker"
ADAPTER_DIR = "/teamspace/studios/this_studio/qwen_resume_ranker_lora"

assert os.path.exists(DATA_PATH), f"File not found: {DATA_PATH} — upload it via the file browser first"
print(f"Data found: {DATA_PATH}")
print(f"Checkpoints will save to: {OUTPUT_DIR}")
print(f"Final adapter will save to: {ADAPTER_DIR}")

## 3. Load and split data (80/20 stratified by role)

In [ ]:
import json
import random
from collections import defaultdict

random.seed(42)

records = []
with open(DATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

by_role = defaultdict(list)
for r in records:
    by_role[r["jd_role_key"]].append(r)

train_records, val_records = [], []
for role, recs in by_role.items():
    random.shuffle(recs)
    split = int(len(recs) * 0.8)
    train_records.extend(recs[:split])
    val_records.extend(recs[split:])

random.shuffle(train_records)
random.shuffle(val_records)

print(f"Train: {len(train_records)}  Val: {len(val_records)}")
print(f"Roles in train: {len(set(r['jd_role_key'] for r in train_records))}")
print(f"Roles in val:   {len(set(r['jd_role_key'] for r in val_records))}")

## 4. Prompt formatter

In [ ]:
SYSTEM_PROMPT = (
    "You are a resume analyst. "
    "Given a job description and a candidate's resume text, "
    "extract the candidate profile and score their fit for the role. "
    "Respond with a single valid JSON object — no markdown, no explanation."
)

def make_user_message(rec: dict) -> str:
    jd = rec["jd"]
    jd_block = json.dumps({
        "job_title":             jd.get("job_title"),
        "domain":                jd.get("domain"),
        "min_experience_years":  jd.get("min_experience_years"),
        "core_technical_skills": jd.get("core_technical_skills"),
        "is_management_role":    jd.get("is_management_role"),
        "education_requirement": jd.get("education_requirement"),
    }, ensure_ascii=False)
    return (
        f"JD Role: {rec['jd_role_key']}\n"
        f"JD:\n{jd_block}\n\n"
        f"Resume:\n{rec['resume_text']}"
    )

def make_assistant_message(rec: dict) -> str:
    output = {
        "candidate":       rec["candidate"],
        "score_breakdown": rec["score_breakdown"],
        "relevance_score": rec["relevance_score"],
        "score_rationale": rec["score_rationale"],
    }
    return json.dumps(output, ensure_ascii=False)

def format_chat(rec: dict) -> dict:
    return {
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": make_user_message(rec)},
            {"role": "assistant", "content": make_assistant_message(rec)},
        ]
    }

sample = format_chat(train_records[0])
print("--- USER (first 400 chars) ---")
print(sample["messages"][1]["content"][:400])
print("\n--- ASSISTANT (first 400 chars) ---")
print(sample["messages"][2]["content"][:400])

## 5. Build HuggingFace datasets

In [ ]:
from datasets import Dataset

train_ds = Dataset.from_list([format_chat(r) for r in train_records])
val_ds   = Dataset.from_list([format_chat(r) for r in val_records])

print(train_ds)
print(val_ds)

## 6. Load model with Unsloth QLoRA (4-bit)

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/Qwen2.5-3B-Instruct",
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r               = 16,
    target_modules  = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    lora_alpha      = 16,
    lora_dropout    = 0.05,
    bias            = "none",
    use_gradient_checkpointing = "unsloth",
    random_state    = 42,
)

print(model.print_trainable_parameters())

## 7. Tokenize with chat template

In [ ]:
from unsloth.chat_templates import get_chat_template
import numpy as np

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

def tokenize(examples):
    texts = [
        tokenizer.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=False,
        )
        for msgs in examples["messages"]
    ]
    return tokenizer(
        texts,
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding=False,
    )

train_tok = train_ds.map(tokenize, batched=True, remove_columns=["messages"])
val_tok   = val_ds.map(tokenize,   batched=True, remove_columns=["messages"])

lengths = [len(x) for x in train_tok["input_ids"]]
print(f"Token lengths — mean: {np.mean(lengths):.0f}  p95: {np.percentile(lengths, 95):.0f}  max: {max(lengths)}")

## 8. Train

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = train_tok,
    eval_dataset  = val_tok,
    args = SFTConfig(
        output_dir                  = OUTPUT_DIR,
        per_device_train_batch_size = 2,
        per_device_eval_batch_size  = 2,
        gradient_accumulation_steps = 4,
        num_train_epochs            = 3,
        warmup_steps                = 40,
        lr_scheduler_type           = "cosine",
        learning_rate               = 2e-4,
        fp16                        = not torch.cuda.is_bf16_supported(),
        bf16                        = torch.cuda.is_bf16_supported(),
        eval_strategy               = "epoch",
        save_strategy               = "epoch",
        save_total_limit            = 2,          # keep only last 2 checkpoints
        load_best_model_at_end      = True,
        metric_for_best_model       = "eval_loss",
        logging_steps               = 20,
        max_seq_length              = MAX_SEQ_LEN,
        packing                     = True,
        seed                        = 42,
        report_to                   = "none",
    ),
)

trainer_stats = trainer.train()
print(trainer_stats)

## 9. Quick inference test

In [ ]:
FastLanguageModel.for_inference(model)

test_rec = val_records[0]

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": make_user_message(test_rec)},
]

input_ids = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

attention_mask = torch.ones_like(input_ids)

with torch.no_grad():
    output_ids = model.generate(
        input_ids,
        attention_mask = attention_mask,
        max_new_tokens = 800,
        temperature    = 0.1,
        do_sample      = True,
        pad_token_id   = tokenizer.eos_token_id,
    )

generated = tokenizer.decode(
    output_ids[0][input_ids.shape[1]:],
    skip_special_tokens=True
)
print("=== GENERATED ===")
print(generated[:1500])
print("\n=== GROUND TRUTH ===")
print(make_assistant_message(test_rec)[:1500])

## 10. Score breakdown evaluation (MAE per axis)

In [ ]:
from collections import defaultdict

SCORE_AXES = [
    "skill_coverage", "experience_fit", "role_alignment",
    "domain_alignment", "education_fit", "deployment_alignment",
    "management_alignment"
]

errors          = defaultdict(list)
relevance_errors = []
json_parse_fails = 0
EVAL_N           = min(100, len(val_records))

for rec in val_records[:EVAL_N]:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": make_user_message(rec)},
    ]
    input_ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    attention_mask = torch.ones_like(input_ids)

    with torch.no_grad():
        out = model.generate(
            input_ids,
            attention_mask = attention_mask,
            max_new_tokens = 800,
            temperature    = 0.1,
            do_sample      = True,
            pad_token_id   = tokenizer.eos_token_id,
        )

    text = tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True)

    try:
        pred           = json.loads(text.strip())
        pred_breakdown = pred.get("score_breakdown", {})
        pred_score     = pred.get("relevance_score", None)
        gt_breakdown   = rec["score_breakdown"]
        gt_score       = rec["relevance_score"]

        for axis in SCORE_AXES:
            if axis in pred_breakdown and axis in gt_breakdown:
                errors[axis].append(abs(pred_breakdown[axis] - gt_breakdown[axis]))
        if pred_score is not None:
            relevance_errors.append(abs(pred_score - gt_score))

    except (json.JSONDecodeError, TypeError, KeyError):
        json_parse_fails += 1

print(f"\nEval on {EVAL_N} val records  |  JSON parse fails: {json_parse_fails}/{EVAL_N}")
print(f"{'Axis':<25} {'MAE':>6}")
print("-" * 33)
for axis in SCORE_AXES:
    if errors[axis]:
        print(f"{axis:<25} {np.mean(errors[axis]):.4f}")
    else:
        print(f"{axis:<25} {'n/a':>6}")
if relevance_errors:
    print(f"{'relevance_score (0-10)':<25} {np.mean(relevance_errors):.4f}")

## 11. Save LoRA adapter
Saved to Lightning persistent storage — survives session end.

In [ ]:
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"LoRA adapter saved to {ADAPTER_DIR}")
print("Files:")
import os
for f in os.listdir(ADAPTER_DIR):
    size = os.path.getsize(os.path.join(ADAPTER_DIR, f))
    print(f"  {f}  ({size/1024/1024:.1f} MB)")